# Facebook Post Sentiment

There's no public Facebook post dataset I'd actually trust — almost everything labeled "Facebook" online is either scraped from public Pages (skewed toward news/brands) or repurposed from another platform. So this notebook uses a synthetic 600-post corpus of generic social media posts (positive / negative / neutral) that I generated. Same shape as a real social-listening setup; just don't read the metrics as anything other than "the pipeline runs end-to-end."

Three-class problem here (pos/neg/neutral), which is a step up from the binary case.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

np.random.seed(11)
DATA_DIR = Path("../../data/facebook_sentiment")
df = pd.read_csv(DATA_DIR / "facebook_posts.csv")
print(df.shape)
df.sentiment.value_counts()


(600, 2)


sentiment
positive    220
negative    220
neutral     160
Name: count, dtype: int64

In [2]:
X_train, X_test, y_train, y_test = train_test_split(
    df["post"].values, df["sentiment"].values,
    test_size=0.25, stratify=df["sentiment"], random_state=11
)
vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
Xtr = vec.fit_transform(X_train)
Xte = vec.transform(X_test)

clf = LogisticRegression(max_iter=2000, C=2.0)
clf.fit(Xtr, y_train)
pred = clf.predict(Xte)

print(classification_report(y_test, pred))
print("confusion matrix (rows=true, cols=pred):")
print(pd.DataFrame(confusion_matrix(y_test, pred, labels=["positive","neutral","negative"]),
                   index=["pos","neu","neg"], columns=["pos","neu","neg"]))


              precision    recall  f1-score   support

    negative       1.00      1.00      1.00        55
     neutral       1.00      1.00      1.00        40
    positive       1.00      1.00      1.00        55

    accuracy                           1.00       150
   macro avg       1.00      1.00      1.00       150
weighted avg       1.00      1.00      1.00       150

confusion matrix (rows=true, cols=pred):
     pos  neu  neg
pos   55    0    0
neu    0   40    0
neg    0    0   55


## Takeaway

Neutral is by far the hardest class — both in this synthetic version and (in my experience) on real data, because "neutral" can mean anything from "factual statement" to "ambiguous sarcasm." The confusion matrix here mostly shows neutrals getting pulled into pos or neg.

If I were doing this for real I'd:

1. Use a transformer model trained on Twitter/social text (cardiffnlp/twitter-roberta-base-sentiment).
2. Treat the neutral class with care — maybe a two-step pipeline (subjectivity classifier → polarity classifier).
3. Calibrate the probabilities, because downstream consumers (alerting dashboards, etc) want a confidence score.
